In [ ]:
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

# Desabilitar paralelismo e progresso
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

# ============================================
# 1. Configurar chave e modelo
# ============================================
#model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_name = "BAAI/bge-m3"

print("Carregando modelo de embeddings...")
embedding_function = HuggingFaceEmbeddings(
    model_name=,model_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True, 'show_progress_bar': False}
)
print("Modelo carregado!")

# ============================================
# 2. Carregar e limpar o dataset
# ============================================

df = pd.read_csv('DADOS_ABERTOS_MEDICAMENTOS.csv', sep=';', encoding='iso-8859-1')

print(f"Número de registros antes da limpeza: {len(df)}")

# ============================================
# 3. Converter linhas em textos semânticos
# ============================================

def row_to_text(row):
    return (
        f"TIPO DO PRODUTO: {row['TIPO_PRODUTO']}\n"
        f"NOME DO PRODUTO: {row['NOME_PRODUTO']}\n"
        f"PRINCÍPIO ATIVO: {row['PRINCIPIO_ATIVO']}\n"
        f"CLASSE TERAPÊUTICA: {row['CLASSE_TERAPEUTICA']}\n"
        f"CATEGORIA REGULATÓRIA: {row['CATEGORIA_REGULATORIA']}\n"
        f"EMPRESA DETENTORA DO REGISTRO: {row['EMPRESA_DETENTORA_REGISTRO']}\n"
        f"NÚMERO DO REGISTRO: {row['NUMERO_REGISTRO_PRODUTO']}\n"
        f"DATA DE FINALIZAÇÃO DO PROCESSO: {row['DATA_FINALIZACAO_PROCESSO']}\n"
        f"DATA DE VENCIMENTO DO REGISTRO: {row['DATA_VENCIMENTO_REGISTRO']}\n"
        f"SITUAÇÃO DO REGISTRO: {row['SITUACAO_REGISTRO']}\n"
        f"NÚMERO DO PROCESSO: {row['NUMERO_PROCESSO']}"
    )

documents = []
for _, row in df.iterrows():
    text = row_to_text(row)
    metadata = {
        "source": "DADOS_ABERTOS_MEDICAMENTOS",
        "NOME_PRODUTO": str(row["NOME_PRODUTO"]).strip(),
        "PRINCIPIO_ATIVO": str(row["PRINCIPIO_ATIVO"]).strip(),
        "CLASSE_TERAPEUTICA": str(row["CLASSE_TERAPEUTICA"]).strip(),
        "EMPRESA": str(row["EMPRESA_DETENTORA_REGISTRO"]).strip(),
    }
    documents.append(Document(page_content=text, metadata=metadata))

print(f"{len(documents)} documentos criados para o vector store.")

# ============================================
# 4. Criar vector store com embeddings
# ============================================

print("Criando vector store (isso pode demorar alguns minutos)...")
db = Chroma.from_documents(
    documents,
    embedding_function,
    persist_directory="../app/data/vectorstores/medicacoes_db"
)

print("✅ Vector store criado com embeddings!")

Carregando modelo de embeddings...


model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

  2025-11-20T01:08:13.298174Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x1099e1d00>), traceback: Some(<traceback object at 0x1202926c0>) }, caller: "src/progress_update.rs:313"
    at /Users/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28

  2025-11-20T01:08:13.330455Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x1099e1d00>), traceback: Some(<traceback object at 0x120292800>) }, caller: "src/progress_update.rs:313"
    at /Users/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28

  2025-11-20T01:08:13.372533Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x1099e1d00>), traceback: Some(<traceback object at 0x1202929c0>) }, caller: "src/progress_update.rs:313"
    at /Users/runner

In [2]:
# ============================================
# 5. Teste de busca
# ============================================

results = db.similarity_search("Ozempic", k=3)
print("\nResultados da busca por 'Ozempic':")
for doc in results:
    print("="*60)
    print(doc.page_content)


Resultados da busca por 'Ozempic':
TIPO DO PRODUTO: MEDICAMENTO
NOME DO PRODUTO: Ozempic
PRINCÍPIO ATIVO: SEMAGLUTIDA
CLASSE TERAPÊUTICA: ANTIDIABETICOS
CATEGORIA REGULATÓRIA: BIOLÓGICO
EMPRESA DETENTORA DO REGISTRO: 82277955000155 - NOVO NORDISK FARMACÊUTICA DO BRASIL LTDA
NÚMERO DO REGISTRO: 117660036.0
DATA DE FINALIZAÇÃO DO PROCESSO: 06/08/2018
DATA DE VENCIMENTO DO REGISTRO: 01/08/2028
SITUAÇÃO DO REGISTRO: VÁLIDO
NÚMERO DO PROCESSO: 25351658916201751
TIPO DO PRODUTO: MEDICAMENTO
NOME DO PRODUTO: OLMESIP
PRINCÍPIO ATIVO: BESILATO DE ANLODIPINO +  OLMESARTANA MEDOXOMILA
CLASSE TERAPÊUTICA: ANTI-HIPERTENSIVOS-ASSOCIACOES MEDICAMENTOSAS
CATEGORIA REGULATÓRIA: SIMILAR
EMPRESA DETENTORA DO REGISTRO: 72593791000111 - NOVA QUIMICA FARMACÊUTICA S/A
NÚMERO DO REGISTRO: 126750291.0
DATA DE FINALIZAÇÃO DO PROCESSO: 14/08/2017
DATA DE VENCIMENTO DO REGISTRO: 01/08/2027
SITUAÇÃO DO REGISTRO: VÁLIDO
NÚMERO DO PROCESSO: 25351238925201458
TIPO DO PRODUTO: MEDICAMENTO
NOME DO PRODUTO: OLZICAR ANL